# Comparison of runtimes

In [1]:
from timeit import timeit
from dataclasses import dataclass
from mandelbrot_calculator import MandelbrotSet

from typing import List

Let's make a Python equivalent to the Rust struct:

In [2]:
class MandelbrotSetTest:
    def __init__(self, grid_size: int) -> None:
        self.grid_size = grid_size

    def make_grid(
            self,
            re_min: float,
            re_max: float,
            im_min: float,
            im_max: float,
            max_iter: int,
    ) -> List[List[int]]:
            grid = [[0 for _ in range(self.grid_size)] for _ in range(self.grid_size)]

            #[allow(clippy::needless_range_loop)]
            for row in range(self.grid_size):
                for col in range(self.grid_size):
                    re = re_min + (col / (self.grid_size - 1)) * (re_max - re_min)
                    im = im_min + (row / (self.grid_size - 1)) * (im_max - im_min)

                    grid[self.grid_size - row - 1][col] = self._iteration_at_which_point_explodes(re, im, max_iter)

            return grid

    @staticmethod
    def _iteration_at_which_point_explodes(re: float, im: float, max_iter: int) -> int:
        z_re = 0.0
        z_im = 0.0

        for i in range(max_iter):
            z_re_new = z_re * z_re - z_im * z_im + re
            z_im_new = 2.0 * z_re * z_im + im

            z_re = z_re_new
            z_im = z_im_new

            # If |z| > 2 at any iteration, escape and return that iteration
            if z_re * z_re + z_im * z_im > 4.0:
                return i

        return max_iter

In [3]:
@dataclass
class Parameters:
    re_min: float
    re_max: float
    im_min: float
    im_max: float
    max_iter: int

## Python vs. Rust (and Rust vs. Rust)

In [4]:
grid_size = 2500
parameters = Parameters(-1, 1, -1, 1, 250)

In [ ]:
t1 = timeit(lambda: MandelbrotSetTest(grid_size).make_grid(**parameters.__dict__), number=10)

In [ ]:
t2 = timeit(lambda: MandelbrotSet(grid_size).make_grid(**parameters.__dict__), number=10)

In [ ]:
t3 = timeit(lambda: MandelbrotSet(grid_size).make_grid_parallell(**parameters.__dict__), number=10)

In [ ]:
print(f"Rust is {t1/t2:.1f} times faster than Python with sequential processing")

Rust is  26.7 times faster than Python (sequential)


In [ ]:
print(f"Rust with parallell processing is {t1/t3:.1f} times faster than Python and {t2/t3:.1f} faster than Rust with sequential processing")

Rust with parallell processing is  110.2 times faster than Python and  4.1 faster than Rust with sequential processing
